In [ ]:
import joblib
from collections import Counter
from pprint import pprint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(context='poster')

In [ ]:
# Load checkpoint (works while benchmark is still running)
ckpt_results, done_set = joblib.load('results/optuna_models_ckpt.joblib')

database = pd.read_json('database.json').T
database.loc[:, 'nrow'] = np.minimum(10000, database['nrow'])

svc_results, log_reg_results, random_forest_results, evaluated_datasets, baseline_times = joblib.load(
    'results/compare_baseline_models.joblib'
)
baseline_datasets = list(evaluated_datasets)

print(f'Datasets in checkpoint: {len(ckpt_results)}')
print(f'Done (dataset, model) pairs: {len(done_set)}')

In [ ]:
FOLD_COLS = ['prauc_fold_1', 'prauc_fold_2', 'prauc_fold_3', 'prauc_fold_4']

# Datasets where TabPFN produced valid (non-NaN) results
tabpfn_datasets = [
    ds for ds, v in ckpt_results.items()
    if 'tabpfn' in v and not any(np.isnan(s) for s in v['tabpfn']['scores'])
]
print(f'Datasets with valid TabPFN results: {len(tabpfn_datasets)}')

excluded = [
    ds for ds, v in ckpt_results.items()
    if 'tabpfn' in v and any(np.isnan(s) for s in v['tabpfn']['scores'])
]
print(f'Excluded (>10 classes, TabPFN NaN): {excluded}')

In [ ]:
OPTUNA_MODELS = {
    'svc':            'SVC (GridSearch)',
    'logreg':         'Logistic Regression (GridSearch)',
    'random_forest':  'Random Forest (Optuna)',
    'xgboost':        'XGBoost (Optuna)',
    'sgd':            'SGD (Optuna)',
    'catboost':       'CatBoost (Optuna)',
    'lgbm':           'LightGBM (Optuna)',
    'lgbm_linear':    'LightGBM Linear (Optuna)',
    'hgb':            'HistGradientBoosting (Optuna)',
    'tabnet':         'TabNet (Optuna)',
    'ft_transformer': 'FT-Transformer (Optuna)',
    'resnet':         'ResNet (Optuna)',
    'tabpfn':         'TabPFN (GridSearch)',
    'tabicl':         'TabICL (GridSearch)',
}

baseline_index = {ds: i for i, ds in enumerate(baseline_datasets)}

rows = []
for ds in tabpfn_datasets:
    # baseline models (from compare_baseline_models.joblib)
    if ds in baseline_index:
        idx = baseline_index[ds]
        for scores, label in [
            (svc_results[idx], 'SVC'),
            (log_reg_results[idx], 'Logistic Regression'),
            (random_forest_results[idx], 'Random Forest'),
        ]:
            rows.append({'dataset': ds, 'model': label, **dict(zip(FOLD_COLS, scores))})

    # optuna + tabpfn models (from checkpoint)
    if ds in ckpt_results:
        for key, label in OPTUNA_MODELS.items():
            if key in ckpt_results[ds]:
                scores = ckpt_results[ds][key]['scores']
                if not any(np.isnan(s) for s in scores):
                    rows.append({'dataset': ds, 'model': label, **dict(zip(FOLD_COLS, scores))})

results_df = pd.DataFrame(rows)
print(f'Total rows: {len(results_df)}')
print(f'Models: {sorted(results_df["model"].unique())}')

In [ ]:
SPLIT_COLS = ['prauc_split_1', 'prauc_split_2', 'prauc_split_3', 'prauc_split_4']
results_df.columns = ['dataset', 'model'] + SPLIT_COLS

# join database metadata
for col in database.columns:
    results_df[col] = results_df['dataset'].map(database[col]).to_numpy()

results_df['mean_prauc'] = results_df[SPLIT_COLS].mean(axis=1)
results_df['min_prauc']  = results_df[SPLIT_COLS].min(axis=1)
results_df['max_prauc']  = results_df[SPLIT_COLS].max(axis=1)
results_df['std_prauc']  = results_df[SPLIT_COLS].std(axis=1)

# Per-fold delta relative to RF baseline
rf_lookup = {}
for ds in tabpfn_datasets:
    if ds in baseline_index:
        idx = baseline_index[ds]
        rf_lookup[ds] = random_forest_results[idx]

RF_COLS    = ['rf_prauc_split_1', 'rf_prauc_split_2', 'rf_prauc_split_3', 'rf_prauc_split_4']
DELTA_COLS = ['delta_split_1', 'delta_split_2', 'delta_split_3', 'delta_split_4']
for i, (rf_col, split_col, delta_col) in enumerate(zip(RF_COLS, SPLIT_COLS, DELTA_COLS)):
    results_df[rf_col]    = results_df['dataset'].map(lambda ds: rf_lookup.get(ds, [np.nan]*4)[i])
    results_df[delta_col] = results_df[split_col] - results_df[rf_col]

results_df['mean_delta'] = results_df[DELTA_COLS].mean(axis=1)
results_df['std_delta']  = results_df[DELTA_COLS].std(axis=1)

# drop datasets where all models are > 0.99 (trivially easy)
results_df = results_df.groupby('dataset').filter(lambda x: x['mean_prauc'].min() < 0.99)
results_df = results_df.reset_index(drop=True)

active_datasets = set(results_df['dataset'])
print(f'Datasets after filtering trivially-easy: {len(active_datasets)}')

In [ ]:
print(f'Average delta PR AUC vs RF baseline (TabPFN-eligible datasets only):')
print(results_df.groupby('model')['mean_delta'].mean().sort_values(ascending=False).to_string())
print()
print(f'Median delta PR AUC vs RF baseline:')
print(results_df.groupby('model')['mean_delta'].median().sort_values(ascending=False).to_string())

In [ ]:
winning_algorithms = []
for dataset in active_datasets:
    df_sub = results_df[results_df['dataset'] == dataset]
    highest_prauc = df_sub['mean_prauc'].max()
    winning_algorithms.extend(df_sub.loc[df_sub['mean_prauc'] >= highest_prauc * 0.995, 'model'])
print(f'Number of datasets each algorithm does best on (out of {len(active_datasets)}):')
pprint(Counter(winning_algorithms))

# Rank distribution
pivot = results_df.pivot_table(index='dataset', columns='model', values='mean_prauc')
models_ordered = results_df.groupby('model')['mean_delta'].mean().sort_values(ascending=False).index.tolist()
rank_pivot = pivot[models_ordered].rank(axis=1, ascending=False, method='average')
rank_dist = pd.DataFrame(index=models_ordered)
for r in range(1, len(models_ordered) + 1):
    rank_dist[f'rank_{r}'] = (rank_pivot == r).sum()
rank_dist['mean_rank'] = rank_pivot.mean()
print()
print('Rank distribution (mean rank):')
print(rank_dist['mean_rank'].sort_values().to_string())

In [ ]:
# Box plot: delta PR AUC vs RF baseline
models_ordered = results_df.groupby('model')['mean_delta'].mean().sort_values(ascending=False).index.tolist()
plot_df = results_df[results_df['model'].isin(models_ordered)].copy()
plot_df['model'] = pd.Categorical(plot_df['model'], categories=models_ordered[::-1], ordered=True)

fig, ax = plt.subplots(figsize=(16, 9))
sns.boxenplot(data=plot_df, x='mean_delta', y='model', ax=ax)
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Mean PR AUC − RF baseline  (positive = better than RF)')
ax.set_title(f'Model performance on TabPFN-eligible datasets (≤10 classes, n={len(active_datasets)})')
plt.tight_layout()
plt.savefig('results/tabpfn_delta_boxen.png', dpi=120)
plt.show()

In [ ]:
# Rank heatmap
rank_frac = pd.DataFrame(index=models_ordered)
n_ds = rank_pivot.notna().all(axis=1).sum()
for r in range(1, len(models_ordered) + 1):
    rank_frac[r] = (rank_pivot[models_ordered] == r).sum() / max(n_ds, 1)

fig, ax = plt.subplots(figsize=(20, 9))
sns.heatmap(rank_frac, annot=True, fmt='.2f', cmap='YlOrRd_r', ax=ax,
            cbar_kws={'label': 'fraction of datasets'})
ax.set_xlabel('Rank (1 = best)')
ax.set_ylabel('Model')
ax.set_title('Rank distribution across TabPFN-eligible datasets')
plt.tight_layout()
plt.savefig('results/tabpfn_rank_heatmap.png', dpi=120)
plt.show()

In [ ]:
# TabPFN vs top competitors: scatter per dataset
competitors = ['CatBoost (Optuna)', 'LightGBM (Optuna)', 'XGBoost (Optuna)', 'Random Forest (Optuna)']
pivot_prauc = results_df.pivot_table(index='dataset', columns='model', values='mean_prauc')

fig, axes = plt.subplots(1, len(competitors), figsize=(20, 5), sharey=False)
for ax, comp in zip(axes, competitors):
    if comp not in pivot_prauc.columns or 'TabPFN (GridSearch)' not in pivot_prauc.columns:
        ax.set_visible(False)
        continue
    sub = pivot_prauc[['TabPFN (GridSearch)', comp]].dropna()
    ax.scatter(sub[comp], sub['TabPFN (GridSearch)'], alpha=0.7)
    lims = [min(sub.min()), max(sub.max())]
    ax.plot(lims, lims, 'k--', linewidth=1)
    ax.set_xlabel(comp, fontsize=9)
    ax.set_ylabel('TabPFN (GridSearch)', fontsize=9)
    n_win = (sub['TabPFN (GridSearch)'] > sub[comp]).sum()
    ax.set_title(f'TabPFN wins: {n_win}/{len(sub)}', fontsize=9)

fig.suptitle('TabPFN vs competitors (mean PR-AUC per dataset)', fontsize=12)
plt.tight_layout()
plt.savefig('results/tabpfn_scatter.png', dpi=120)
plt.show()

In [ ]:
# TabPFN delta vs nrow
tabpfn_df = results_df[results_df['model'] == 'TabPFN (GridSearch)'].copy()

fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(tabpfn_df['nrow'], tabpfn_df['mean_delta'], alpha=0.7)
ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.set_xscale('log')
ax.set_xlabel('Number of rows (log scale)')
ax.set_ylabel('TabPFN mean delta PR AUC vs RF baseline')
ax.set_title('TabPFN relative performance vs dataset size')
for _, row in tabpfn_df[tabpfn_df['mean_delta'].abs() > 0.05].iterrows():
    ax.annotate(row['dataset'], (row['nrow'], row['mean_delta']), fontsize=7,
                xytext=(4, 4), textcoords='offset points')
plt.tight_layout()
plt.savefig('results/tabpfn_delta_vs_nrow.png', dpi=120)
plt.show()